# MCP server 3 — Failure-mode catalog

Read stored failure modes and optionally use an LLM to propose additional modes.

**Tutorial contract:** run cells from top to bottom. The notebook discovers the live MCP schema,
does not print secrets, and does not write to the failure-mode database.


## Goal

1. Verify the live FMSR MCP contract.
2. Retrieve stored failure modes from CouchDB using `get_failure_modes`.
3. Optionally generate additional modes using the configured FMSR model.

Failure-mode/sensor mapping is intentionally not exposed by the current server. In the end-to-end
notebook, Stirrup combines IoT sensor evidence with FMSR failure modes in its reasoning.


In [8]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError(
        "Open this notebook from inside the AssetOpsBench repository, "
        "or change the VS Code working directory to the repository root."
    )

REPO = find_repo()
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

In [9]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def fmsr_request(operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "fmsr-mcp-server"],
        cwd=str(REPO),
        env=os.environ.copy(),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def parse_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = text
    return payload

async def list_fmsr_tools():
    response = await fmsr_request("list")
    return [
        {"name": tool.name, "description": tool.description, "schema": tool.inputSchema}
        for tool in response.tools
    ]

async def call_fmsr(name, **arguments):
    payload = parse_result(await fmsr_request("call", name, arguments))
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"FMSR {name} failed: {payload['error']}")
    if isinstance(payload, str) and (
        payload.startswith("Unknown tool:") or payload.startswith("Error executing tool")
    ):
        raise RuntimeError(payload)
    return payload


## 1. Discover and validate the live MCP contract

This prevents stale notebook calls from silently drifting away from the running server.


In [10]:
tools = await list_fmsr_tools()
contract = {
    tool["name"]: list(tool["schema"].get("properties", {}))
    for tool in tools
}
contract


{'get_failure_modes': ['asset_class'],
 'generate_failure_modes': ['asset_class', 'max_modes'],
 'add_failure_modes': ['asset_class', 'failure_modes', 'exhaustive', 'source']}

In [11]:
expected = {
    "get_failure_modes": {"asset_class"},
    "generate_failure_modes": {"asset_class", "max_modes"},
    "add_failure_modes": {"asset_class", "failure_modes", "exhaustive", "source"},
}
for name, parameters in expected.items():
    assert name in contract, f"Required FMSR tool is missing: {name}"
    assert set(contract[name]) == parameters, (
        f"Schema changed for {name}: expected {sorted(parameters)}, "
        f"received {sorted(contract[name])}"
    )
print("FMSR contract is compatible.")


FMSR contract is compatible.


## 2. Retrieve stored failure modes

The parameter is `asset_class`, not `asset_name`. Digits and capitalization are normalized, but an
explicit class such as `chiller` is clearest. This call uses CouchDB and makes no LLM request.


In [13]:
ASSET_CLASS = "pump"

failure_modes = await call_fmsr(
    "get_failure_modes",
    asset_class=ASSET_CLASS,
)

failure_modes

{'asset_class': 'pump',
 'failure_modes': ['seal leakage', 'impeller wear'],
 'exhaustive': False,
 'source': 'synthetic sample'}

## 3. Optional LLM generation 
### Without database write

`generate_failure_modes` proposes modes and returns them without persisting anything. Its provider is
selected by `FMSR_MODEL_ID`. If you changed `.env`, restart the notebook kernel before running this
section so the notebook and newly spawned MCP process receive the same environment.


In [14]:
MODEL_ID = os.getenv(
    "FMSR_MODEL_ID",
    "watsonx/meta-llama/llama-3-3-70b-instruct",
)

if MODEL_ID.startswith("watsonx/"):
    required = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID"]
elif MODEL_ID.startswith("tokenrouter/"):
    required = ["TOKENROUTER_API_KEY", "TOKENROUTER_BASE_URL"]
else:
    required = ["LITELLM_API_KEY", "LITELLM_BASE_URL"]

missing = [name for name in required if not os.getenv(name)]
print("FMSR model:", MODEL_ID)
print("generation preflight:", "ready" if not missing else "missing " + ", ".join(missing))


FMSR model: watsonx/meta-llama/llama-3-3-70b-instruct
generation preflight: ready


In [16]:
RUN_GENERATION = True  # Set True only when you want to make a model call.

if RUN_GENERATION:
    if missing:
        raise RuntimeError("Configure these variables and restart the kernel: " + ", ".join(missing))
    generated = await call_fmsr(
        "generate_failure_modes",
        asset_class=ASSET_CLASS,
        max_modes=5,
    )
    display(generated)
else:
    print("Generation skipped. Set RUN_GENERATION=True to enable it.")


{'asset_class': 'pump',
 'known': ['seal leakage', 'impeller wear'],
 'generated': ['Bearing failure',
  'Motor burnout',
  'Coupling misalignment',
  'Packing leakage',
  'Cavitation'],
 'failure_modes': ['seal leakage',
  'impeller wear',
  'Bearing failure',
  'Motor burnout',
  'Coupling misalignment',
  'Packing leakage',
  'Cavitation'],
 'source': 'LLM:watsonx/meta-llama/llama-3-3-70b-instruct',
 'message': "generated 5 new failure mode(s) for asset_class 'pump' using 2 stored mode(s) as context; nothing was persisted."}

## Supported scope

This server exposes stored failure-mode lookup, optional LLM generation, and persistent
catalog updates. This tutorial uses only the read-only tools. Sensor mapping is not part
of the current FMSR MCP contract, and `add_failure_modes` is not executed because it
writes to CouchDB.

## Takeaway

You exercised the current FMSR server over MCP stdio, validated its schema, performed a grounded
catalog lookup, and kept optional LLM generation separate from persistent writes.
    